# 00 — Filter IT Jobs

Filters `final_jobs_dataset.csv` (1,068 rows) down to IT-only postings using:
1. Title-based role pattern matching (11 IT role groups)
2. Exclusion patterns for clearly non-IT roles
3. Tech text scoring fallback for ambiguous titles

**Outputs:**
- `data/processed/jobs/final_jobs_dataset_it_only.csv` — IT-only rows with role labels
- `data/processed/jobs/it_job_filter_audit.csv` — all rows with decision, role, reason, score
- `data/processed/jobs/it_job_filter_review_queue.csv` — borderline cases for manual review

In [ ]:
import csv
import re
from collections import Counter
from pathlib import Path

# --- root detection ---
_nb_path = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
if _nb_path:
    import os; os.chdir(Path(_nb_path).resolve().parent.parent.parent)
elif not (Path.cwd() / 'data').exists():
    _root = Path.cwd()
    for _ in range(4):
        if (_root / 'data').exists(): break
        _root = _root.parent
    import os; os.chdir(_root)
assert (Path.cwd() / 'data').exists(), f'Cannot find project root from {Path.cwd()}'

ROOT = Path.cwd()
INPUT_CSV       = ROOT / 'data/processed/jobs/final_jobs_dataset.csv'
OUTPUT_IT_CSV   = ROOT / 'data/processed/jobs/final_jobs_dataset_it_only.csv'
OUTPUT_AUDIT_CSV  = ROOT / 'data/processed/jobs/it_job_filter_audit.csv'
OUTPUT_REVIEW_CSV = ROOT / 'data/processed/jobs/it_job_filter_review_queue.csv'
print('Root:', ROOT)

## Pattern definitions

In [ ]:
ROLE_PATTERNS: list[tuple[str, list[str]]] = [
    (
        'Backend',
        [
            r'\bbackend\b', r'\bback\s*-?end\b', r'\bsoftware engineer\b', r'\bsoftware developer\b',
            r'\bpython developer\b', r'\bpython engineer\b', r'\bjava developer\b', r'\bjava engineer\b',
            r'\b\.net developer\b', r'\b\.net engineer\b', r'\bc# developer\b', r'\bc# engineer\b',
            r'\bnode(?:\.js|js)?\b', r'\bphp\b', r'\bgolang\b', r'\brust engineer\b', r'\bruby\b',
            r'\bscala\b', r'\bkernel developer\b', r'\bapi developer\b', r'\bhtml/markup developer\b',
            r'\bmarkup developer\b', r'\bwebflow developer\b', r'\bdelphi developer\b', r'\blaravel\b',
            r'\bsapui5\b', r'\bfiori developer\b', r'\bhtml5 game developer\b',
            r'\bc\s*/\s*c\+\+\s+engineer\b', r'\bunity developer\b', r'\bsearch engineer\b',
            r'\bintegrations engineer\b', r'\bautomation engineer\b', r'\bproduct engineer\b',
        ],
    ),
    (
        'Frontend / JS',
        [
            r'\bfrontend\b', r'\bfront\s*-?end\b', r'\breact\b', r'\bangular\b', r'\bvue\b',
            r'\bjavascript\b', r'\btypescript\b', r'\bui developer\b', r'\bweb developer\b',
        ],
    ),
    ('Full Stack', [r'\bfull\s*-?stack\b', r'\bfullstack\b']),
    (
        'Mobile',
        [
            r'\bandroid\b', r'\bios\b', r'\bmobile developer\b', r'\bmobile engineer\b',
            r'\bflutter\b', r'\breact native\b', r'\bswift\b', r'\bkotlin\b',
        ],
    ),
    (
        'Data / ML / AI',
        [
            r'\bdata engineer\b', r'\bdata scientist\b', r'\bdata analyst\b', r'\bml engineer\b',
            r'\bmachine learning\b', r'\bartificial intelligence\b', r'\bai engineer\b', r'\bgenai\b',
            r'\bnlp\b', r'\bcomputer vision\b', r'\bbi developer\b', r'\bpower bi developer\b',
            r'\bbi engineer\b', r'\bdata specialist\b', r'\bdata quality analyst\b', r'\bdata ops\b',
            r'\bsystem analyst\b', r'\bbusiness systems analyst\b', r'\bdata architect\b',
            r'\benterprise architecture\b',
        ],
    ),
    (
        'DevOps / Cloud',
        [
            r'\bdevops\b', r'\bsre\b', r'\bsite reliability\b', r'\bcloud\b',
            r'\bplatform engineer\b', r'\bplatform release engineer\b', r'\brelease engineer\b',
            r'\bdeployment engineer\b', r'\binfrastructure\b', r'\bbuild engineer\b',
            r'\bdatabase reliability engineer\b', r'\bdbre\b',
        ],
    ),
    (
        'QA / Testing',
        [
            r'\bqa\b', r'\bquality assurance\b', r'\btest automation\b', r'\btester\b',
            r'\bsdet\b', r'\bsqa\b', r'\baqa\b', r'\bquality engineer\b', r'\btest engineer\b',
            r'\bsoftware engineer in test\b', r'\bfunctional testing\b',
        ],
    ),
    (
        'Security',
        [
            r'\bsecurity\b', r'\bcyber', r'\binfosec\b', r'\bsoc analyst\b',
            r'\bpenetration\b', r'\bvulnerability\b', r'\bokta administrator\b',
        ],
    ),
    (
        'IT Support / Admin',
        [
            r'\bit support\b', r'\btechnical support engineer\b', r'\bfrontline support engineer\b',
            r'\bend-user support engineer\b', r'\bapplication support engineer\b',
            r'\bmiddleware support engineer\b', r'\btechnical support team lead\b',
            r'\bservice desk\b', r'\bservicedesk engineer\b', r'\bsystem administrator\b',
            r'\bdatabase administrator\b', r'\bnetwork administrator\b', r'\bjira administrator\b',
            r'\bcorporate applications administrator\b', r'\bgoogle workspace deployment engineer\b',
            r'\bdata center technician\b', r'\bendpoint operations technician\b', r'\bnoc engineer\b',
        ],
    ),
    (
        'Hardware / Embedded',
        [
            r'\bembedded\b', r'\bfirmware\b', r'\bhardware\b', r'\bfpga\b', r'\basic\b',
            r'\bvlsi\b', r'\brtl\b', r'\bverilog\b', r'\bvhdl\b', r'\bpcb\b',
            r'\bphysical design engineer\b', r'\bdesign verification\b', r'\bmbist\b',
            r'\bdft\b', r'\bsta\b', r'\bcad\/methodology\b', r'\banalog layout\b',
            r'\bcharacterization engineer\b', r'\br&d engineer\b',
        ],
    ),
    (
        'Technical Management',
        [
            r'\bengineering manager\b', r'\bdirector,\s*software engineering\b',
            r'\bplatform engineering team lead\b', r'\bengineering technical leader\b',
            r'\bengineering team lead\b', r'\bengineering team leader\b',
            r'\btechnical project manager\b', r'\bbackend team lead\b', r'\bqa team lead\b',
            r'\bteam lead back-end developer\b', r'\bhead of .*enterprise architecture\b',
            r'\bsenior technical manager\b', r'\bsolution architect\b', r'\bsolutions architect\b',
        ],
    ),
]

DROP_PATTERNS = [
    r'\bmarketing\b', r'\bmedia\b', r'\bpublic relations\b', r'\bpr specialist\b',
    r'\bseo\b', r'\bsales\b', r'\baccount manager\b', r'\bbusiness development\b',
    r'\brecruit', r'\bhr\b', r'\bhuman resources\b', r'\btalent acquisition\b',
    r'\bpeople analytics\b', r'\blegal\b', r'\bcompliance\b', r'\bprocurement\b',
    r'\blogistics\b', r'\bfinance\b', r'\bfinancial\b', r'\baccountant\b',
    r'\bbilling\b', r'\bhotel\b', r'\binterior architect\b', r'\binterior designer\b',
    r'\bgraphic\b', r'\billustrator\b', r'\banimator\b', r'\bartist\b',
    r'\bmotion designer\b', r'\bux researcher\b', r'\bproduct designer\b',
    r'\bdesigner\b', r'\bproducer\b', r'\bteacher\b', r'\btrainer\b',
    r'\bcustomer support\b', r'\bsupport representative\b', r'\bsupport specialist\b',
    r'\bmerchant risk analyst\b', r'\bpayroll analyst\b', r'\bchargeback\b',
    r'\bstrategy associate\b', r'\bgrowth specialist\b',
]

STRONG_TECH_HINT_WORDS = [
    'engineer', 'developer', 'administrator', 'architect', 'platform', 'software',
    'system', 'database', 'network', 'security', 'cloud', 'devops', 'qa', 'test',
    'data', 'ml', 'ai', 'frontend', 'backend', 'full stack', 'mobile', 'support',
    'embedded', 'firmware', 'hardware',
]

AMBIGUOUS_BUSINESS_HINTS = [
    'analyst', 'manager', 'product', 'owner', 'consultant', 'specialist', 'audit', 'intern', 'interviewer',
]

TECH_TEXT_TERMS = [
    'python', 'java', 'javascript', 'typescript', 'react', 'angular', 'vue', 'node', 'php',
    'sql', 'database', 'api', 'microservice', 'docker', 'kubernetes', 'aws', 'azure', 'gcp',
    'linux', 'git', 'ci/cd', 'terraform', 'ansible', 'jenkins', 'pytest', 'selenium',
    'playwright', 'postman', 'machine learning', 'data science', 'data engineer', 'power bi',
    'tableau', 'firmware', 'embedded', 'asic', 'fpga', 'vlsi', 'verilog', 'vhdl', 'rtl',
    'soc', 'security', 'encryption', 'network', '.net', 'c#', 'c++', 'golang', 'swift',
    'kotlin', 'android', 'ios', 'jira', 'okta',
]

# pre-compile
COMPILED_ROLE_PATTERNS = [
    (role, [re.compile(p, re.IGNORECASE) for p in patterns])
    for role, patterns in ROLE_PATTERNS
]
COMPILED_DROP_PATTERNS = [re.compile(p, re.IGNORECASE) for p in DROP_PATTERNS]
print(f'{len(COMPILED_ROLE_PATTERNS)} role groups, {sum(len(p) for _, p in COMPILED_ROLE_PATTERNS)} role patterns, {len(COMPILED_DROP_PATTERNS)} drop patterns')

## Classification functions

In [ ]:
def normalize_text(value: str) -> str:
    return ' '.join((value or '').lower().split())

def match_role(title: str) -> str | None:
    for role, patterns in COMPILED_ROLE_PATTERNS:
        if any(p.search(title) for p in patterns):
            return role
    return None

def match_drop_reason(title: str) -> str | None:
    for p in COMPILED_DROP_PATTERNS:
        if p.search(title):
            return p.pattern
    return None

def tech_text_score(text: str) -> int:
    return sum(term in text for term in TECH_TEXT_TERMS)

def has_strong_tech_hint(title: str) -> bool:
    return any(word in title for word in STRONG_TECH_HINT_WORDS)

def has_ambiguous_business_hint(title: str) -> bool:
    return any(word in title for word in AMBIGUOUS_BUSINESS_HINTS)

def classify_job(row: dict) -> tuple[str, str, str, int]:
    title = normalize_text(row.get('job_title', ''))
    text  = normalize_text(row.get('full_text', ''))[:5000]

    role        = match_role(title)
    drop_reason = match_drop_reason(title)
    score       = tech_text_score(text)

    if role:
        if role == 'Technical Management' and score < 2:
            return 'review', role, 'title matched Technical Management but text support was weak', score
        if role == 'Data / ML / AI' and drop_reason:
            return 'review', role, f'title matched {role} but also matched exclusion pattern {drop_reason}', score
        if drop_reason and score < 2 and role not in {'Data / ML / AI', 'Technical Management'}:
            return 'drop', 'Non-IT', f'title matched exclusion pattern {drop_reason}', score
        return 'keep', role, f'title matched {role}', score

    if drop_reason:
        return 'drop', 'Non-IT', f'title matched exclusion pattern {drop_reason}', score

    if score >= 3 and has_strong_tech_hint(title) and has_ambiguous_business_hint(title):
        return 'review', 'Ambiguous', f'title mixed technical and business cues with {score} technical text signals', score
    if score >= 4 and has_strong_tech_hint(title):
        return 'keep', 'General IT', f'text had {score} technical signals', score
    if score >= 2 and has_strong_tech_hint(title):
        return 'review', 'Ambiguous', f'title was technical-ish and text had {score} technical signals', score

    return 'drop', 'Non-IT', f'no strong IT title pattern and only {score} technical text signals', score

## Run classification and save outputs

In [ ]:
with INPUT_CSV.open(newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

audit_rows, it_rows, review_rows = [], [], []
decision_counts: Counter = Counter()
role_counts: Counter = Counter()

for row in rows:
    decision, role, reason, score = classify_job(row)
    enriched = dict(row)
    enriched['it_filter_decision'] = decision
    enriched['it_role_group']      = role
    enriched['it_filter_reason']   = reason
    enriched['it_tech_text_score'] = str(score)
    audit_rows.append(enriched)
    decision_counts[decision] += 1
    role_counts[role] += 1
    if decision == 'keep':   it_rows.append(enriched)
    elif decision == 'review': review_rows.append(enriched)

fieldnames = list(audit_rows[0].keys())
for path, data in [(OUTPUT_AUDIT_CSV, audit_rows), (OUTPUT_IT_CSV, it_rows), (OUTPUT_REVIEW_CSV, review_rows)]:
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

print(f'Input rows : {len(rows)}')
for d, c in decision_counts.most_common():
    print(f'  {d:>6}: {c:4d}  ({c/len(rows)*100:.1f}%)')
print(f'\nIT-only CSV  -> {OUTPUT_IT_CSV.relative_to(ROOT)}  ({len(it_rows)} rows)')
print(f'Audit CSV    -> {OUTPUT_AUDIT_CSV.relative_to(ROOT)}')
print(f'Review CSV   -> {OUTPUT_REVIEW_CSV.relative_to(ROOT)}  ({len(review_rows)} rows)')

## Results summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

it_df = pd.read_csv(OUTPUT_IT_CSV)

role_series = it_df['it_role_group'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
role_series.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Job count')
ax.set_title(f'IT jobs by role group (n={len(it_df)})')
plt.tight_layout()
plt.show()

print('\nRole breakdown:')
for role, count in role_series.items():
    print(f'  {role:<25} {count:3d}  ({count/len(it_df)*100:.1f}%)')